# 08 - Federated HOMOGEN (IID), per-dataset (Paper 4, kontrol eksperimen)

**Tujuan:** menjawab pertanyaan kontrol: *apakah FedAvg bekerja baik ketika
setting-nya HOMOGEN?* Berbeda dari nb03 (lintas-dataset CIC+UNSW, heterogen),
notebook ini menguji **tiap dataset SENDIRI** (CIC-only, lalu UNSW-only),
dipecah **IID** ke K klien. Ini mereplikasi kondisi SOTA (mis. Fed-ANIDS) yang
memecah SATU dataset ke banyak klien.

**Kenapa penting:** ini **kontrol yang hilang** dari run-1/run-2. Bila federated
HOMOGEN mendekati centralized -> pipeline FedAvg+MLP+SFM kita **benar**, dan
kegagalan lintas-dataset (nb03) murni akibat **heterogenitas**, bukan bug.
Bila homogen pun gagal -> ada masalah implementasi yang harus diperbaiki dulu.

**Konsistensi:** reuse `../aws/fed_node.py` (MLP 128,64; FedAvg; z-score) -
IDENTIK dgn nb03 & node EC2. Satu-satunya yang berubah: data homogen + partisi IID.

**Output:** `fed_out/homogen_iid.csv` (centralized/local-only/federated untuk
CIC-only & UNSW-only) + `homogen_iid_curve.csv` (kurva konvergensi).

> Jalankan setelah nb01 (butuh `fed_out/{cic,unsw}_{train,test}.npz`). JANGAN mengarang angka.

## 0. Setup + impor logika FL dari fed_node.py

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import (matthews_corrcoef, recall_score, precision_score,
                             f1_score, balanced_accuracy_score)
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
if AWS_DIR not in sys.path: sys.path.insert(0, AWS_DIR)
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
OUTDIR='fed_out'; os.makedirs(OUTDIR, exist_ok=True)
# Hyperparameter SAMA dgn nb03 (run-2) agar apple-to-apple.
ROUNDS=100; LOCAL_EPOCHS=5; LR=3e-3; SEED=42; K=4
print('fed_node loaded | LAYERS', fed.LAYERS, '| K klien IID =', K)

## 1. Muat partisi per-dataset (dari nb01)

Kita pakai train+test tiap dataset APA ADANYA (sudah di ruang 9 fitur SFM).
Di sini TIDAK ada pencampuran CIC<->UNSW (itu kasus heterogen nb03).

In [ ]:
def load(name):
    p=os.path.join(OUTDIR,name)
    if not os.path.exists(p): print('MISSING',p); return None,None
    d=np.load(p, allow_pickle=True); return np.asarray(d['X'],float), np.asarray(d['y'],int)
data={}
for ds,(ftr,fte) in {'CIC':('cic_train.npz','cic_test.npz'),
                     'UNSW':('unsw_train.npz','unsw_test.npz')}.items():
    Xtr,ytr=load(ftr); Xte,yte=load(fte)
    if Xtr is not None:
        data[ds]=(Xtr,ytr,Xte,yte)
        print(f'{ds}: train {Xtr.shape} pos {round(float(ytr.mean()),3)} | '
              f'test {Xte.shape} pos {round(float(yte.mean()),3)}')
ok=len(data)>0
if not ok: print('(partisi belum ada - jalankan nb01 dulu di SageMaker)')

## 2. Helper: partisi IID + evaluasi + prediksi

Partisi IID = acak merata (shuffle lalu bagi K bagian). Ini kondisi PALING
homogen antar-klien (tiap klien = sampel acak dari distribusi yang sama).

In [ ]:
def iid_split(X, y, k, seed=SEED):
    rng=np.random.RandomState(seed); idx=rng.permutation(len(y))
    parts=np.array_split(idx, k)
    return [(X[p], y[p]) for p in parts]
def metrics(y,yp):
    return dict(MCC=round(matthews_corrcoef(y,yp),4),
                recall=round(recall_score(y,yp,zero_division=0),4),
                precision=round(precision_score(y,yp,zero_division=0),4),
                F1=round(f1_score(y,yp,zero_division=0),4),
                bal_acc=round(balanced_accuracy_score(y,yp),4))
def predict(W, X, mu, sd):
    out,_=fed.forward(W, fed.zscore_apply(X, mu, sd)); return (out>=0.5).astype(int)

## 3. Jalankan 3 baseline per-dataset (centralized / local-only / federated-IID)

Untuk tiap dataset homogen:
- **centralized**: latih di SELURUH train dataset (upper bound).
- **local-only**: tiap klien IID latih di shard-nya; lapor rerata MCC antar-klien.
- **federated-IID**: FedAvg K klien IID (z-score lokal per-klien, identik EC2).
Semua dievaluasi di **test dataset yang SAMA** (homogen; bukan cross-dataset).

In [ ]:
res=[]; curves=[]
if ok:
    for ds,(Xtr,ytr,Xte,yte) in data.items():
        # --- centralized ---
        mu_c,sd_c=fed.zscore_fit(Xtr)
        Wc=fed.init_weights(SEED)
        Wc=fed.train_local(Wc, fed.zscore_apply(Xtr,mu_c,sd_c), ytr,
                           epochs=ROUNDS*LOCAL_EPOCHS, lr=LR, mu=0.0, seed=SEED)
        res.append({'dataset':ds,'setting':'centralized', **metrics(yte, predict(Wc,Xte,mu_c,sd_c))})
        # --- partisi IID ---
        shards=iid_split(Xtr, ytr, K)
        # --- local-only (rerata antar-klien IID) ---
        loc_mcc=[]
        for (Xl,yl) in shards:
            mu_l,sd_l=fed.zscore_fit(Xl)
            Wl=fed.init_weights(SEED)
            Wl=fed.train_local(Wl, fed.zscore_apply(Xl,mu_l,sd_l), yl,
                               epochs=ROUNDS*LOCAL_EPOCHS, lr=LR, mu=0.0, seed=SEED)
            loc_mcc.append(matthews_corrcoef(yte, predict(Wl,Xte,mu_l,sd_l)))
        res.append({'dataset':ds,'setting':'local-only(mean)','MCC':round(float(np.mean(loc_mcc)),4),
                    'recall':'','precision':'','F1':'','bal_acc':''})
        # --- federated-IID (FedAvg) ---
        zc=[fed.zscore_fit(Xl) for (Xl,yl) in shards]
        Xz=[fed.zscore_apply(shards[i][0], *zc[i]) for i in range(K)]
        sizes=[len(shards[i][1]) for i in range(K)]
        Wg=fed.init_weights(SEED)
        mu_g,sd_g=fed.zscore_fit(Xtr)  # z-score monitoring utk evaluasi global homogen
        for t in range(ROUNDS):
            cw=[]; cs=[]
            for i in range(K):
                Wl=fed.train_local(Wg, Xz[i], shards[i][1], epochs=LOCAL_EPOCHS,
                                   lr=LR, mu=0.0, W_global=Wg, seed=SEED)
                cw.append(Wl); cs.append(sizes[i])
            Wg=fed.fedavg(cw, cs)
            yp=predict(Wg, Xte, mu_g, sd_g)
            curves.append({'dataset':ds,'round':t,'MCC':round(matthews_corrcoef(yte,yp),4)})
        res.append({'dataset':ds,'setting':'federated-IID', **metrics(yte, predict(Wg,Xte,mu_g,sd_g))})
        print(f'{ds} selesai: centralized/local/federated dihitung')
print('=== SEL 3 (3 baseline per-dataset) SELESAI ===')

## 4. Simpan hasil + tampilkan

**Ekspektasi (H-kontrol):** untuk tiap dataset HOMOGEN, urutan wajar
`centralized >= federated-IID >= local-only`, dan federated-IID harus **dekat**
ke centralized (gap kecil). Bila terpenuhi -> pipeline FL kita VALID.

In [ ]:
if ok and len(res):
    dfres=pd.DataFrame(res)
    dfres.to_csv(os.path.join(OUTDIR,'homogen_iid.csv'),index=False)
    pd.DataFrame(curves).to_csv(os.path.join(OUTDIR,'homogen_iid_curve.csv'),index=False)
    import IPython.display as ipd; ipd.display(dfres)
    print('\nCEK: federated-IID harus mendekati centralized (gap kecil) di tiap dataset.')
    print('Bandingkan dgn nb03 (heterogen) tempat federated KOLAPS.')
else:
    print('(lewati: partisi belum ada - jalankan nb01)')
print('=== SEL 4 (hasil homogen) SELESAI ===')

## 5. Unggah hasil ke S3 (results/homogen)

In [ ]:
S3_BUCKET=os.environ.get('FED_S3_BUCKET','ssh-detection-features-232032302717')
REGION=os.environ.get('AWS_REGION','ap-southeast-1')
PREFIX='unsw-far/federated/results/homogen'
try:
    import boto3
    s3=boto3.client('s3',region_name=REGION); up=0
    for f in ('homogen_iid.csv','homogen_iid_curve.csv'):
        p=os.path.join(OUTDIR,f)
        if os.path.exists(p):
            s3.upload_file(p, S3_BUCKET, f'{PREFIX}/{f}'); up+=1; print('  ok ->', f)
    print(f'\nunggah {up} berkas -> s3://{S3_BUCKET}/{PREFIX}/')
except Exception as e:
    print('(upload S3 gagal:',e,')')
print('=== SEL 5 (unggah S3) SELESAI ===')

## 6. Catatan untuk naskah

- Hasil ini = **kontrol eksperimen** yang memperkuat argumen Paper 4:
  bila federated-IID HOMOGEN mendekati centralized, maka pipeline FL kita VALID
  dan kolaps di nb03 murni akibat **heterogenitas lintas-dataset** (bukan bug).
- Mereplikasi kondisi SOTA (mis. Fed-ANIDS: satu dataset dipecah ke banyak klien).
- Bandingkan tabel: `homogen_iid.csv` (homogen, nb08) vs `fedavg_sfm.csv`
  (heterogen, nb03) -> kontras inilah bukti utama 'SFM perlu tapi tak cukup'.
- Logika FL identik `aws/fed_node.py` (MLP 128,64; FedAvg) -> apple-to-apple.